# 🚀 Worker GPU Google Colab : Reconstruction 3D Dense (VGGT vs COLMAP) & Benchmark
**Module Vision par Ordinateur — Cycle d'Ingénieur LSI 3 (FST Tanger)**  
**Réalisé par :** Yahya AHMANE & Yassine KAMOUSS — **Encadré par :** Pr. M'hamed AIT KBIR

Ce notebook est calibré spécifiquement pour **Google Colab (Python 3.12 / GPU T4)** :
1. **Zéro conflit NumPy / PyTorch** : utilise directement l'environnement PyTorch CUDA & NumPy pré-installé de Colab (aucune recompilation de `numpy` ou `torch`).
2. **Détection directe de vos fichiers dans `/content`** : lit automatiquement `/content/Ignatius.zip` et `/content/Ignatius.ply` (ou `Truck` / `Barn`).
3. **Reconstruction 3D Dense VGGT (CVPR 2025)** : inférence feed-forward en précision mixte FP16 avec gestion mémoire VRAM T4.
4. **Reconstruction 3D COLMAP Haute Densité** : extraction SIFT haute densité + triangulation multi-vues + densification MVS.
5. **Recalage $Sim(3)$ + ICP & Évaluation Quantitative** : compare les nuages à la référence (**LiDAR Tanks and Temples** et **VGGT**) directement dans Colab.
6. **Export `<SCENE>_bundle.zip`** : archive légère prête à être importée dans votre application locale Streamlit (`app.py`).

In [ ]:
# ==============================================================================
# 1. INSTALLATION RAPIDE SPÉCIFIQUE GOOGLE COLAB (SANS TOUCHER À NUMPY/PYTORCH)
# ==============================================================================
import os, sys, subprocess
!nvidia-smi

# 1.1 Réparation automatique si une ancienne exécution avait rétrogradé NumPy
chk = subprocess.run([sys.executable, '-c', 'import numpy, scipy, pandas'], capture_output=True, text=True)
if chk.returncode != 0:
    print('[*] Réalignement de NumPy avec le noyau Google Colab Python 3.12...')
    !pip install -q "numpy>=2.0,<2.1" --upgrade
    print('🔄 Redémarrage automatique du processus Python pour appliquer NumPy proprement...')
    os.kill(os.getpid(), 9)

# 1.2 Installation de COLMAP (apt) et d'Open3D sans modifier les paquets système de Colab
!apt-get update -qq && apt-get install -y -qq colmap xvfb
import numpy as np
NP_VER = np.__version__
!pip install -q --no-warn-conflicts "numpy=={NP_VER}" open3d einops safetensors timm huggingface_hub

# 1.3 Clonage direct de VGGT (sans 'pip install' pour ne jamais écraser le PyTorch/NumPy de Colab)
if not os.path.exists('/content/vggt'):
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git /content/vggt
if '/content/vggt' not in sys.path:
    sys.path.insert(0, '/content/vggt')

print(f'[OK] Environnement Colab prêt en quelques secondes (NumPy {np.__version__}, PyTorch CUDA actif).')

In [ ]:
# ==============================================================================
# 2. CONFIGURATION DE LA SCÈNE ET VÉRIFICATION DES FICHIERS UPLOADÉS
# ==============================================================================
import os, sys, time, json, shutil, zipfile
from pathlib import Path
import numpy as np
import torch
import open3d as o3d
from scipy.spatial import cKDTree

if '/content/vggt' not in sys.path:
    sys.path.insert(0, '/content/vggt')

# 👉 SCÈNE ACTIVE : 'Ignatius', 'Truck' ou 'Barn'
SCENE_NAME = 'Ignatius'
# Nombre de vues échantillonnées à 360° (30 vues = optimal pour les 15 Go VRAM du GPU T4)
NUM_VIEWS = 30

TAU_MM_MAP = {'Ignatius': 3.0, 'Truck': 5.0, 'Barn': 10.0}
VOXEL_M_MAP = {'Ignatius': 0.005, 'Truck': 0.010, 'Barn': 0.015}
TAU_MM = TAU_MM_MAP.get(SCENE_NAME, 3.0)
VOXEL_M = VOXEL_M_MAP.get(SCENE_NAME, 0.005)

WORK_DIR = Path(f'/content/workspace/{SCENE_NAME}')
IMG_DIR = WORK_DIR / 'images'
OUT_DIR = WORK_DIR / 'outputs'
RAW_DIR = WORK_DIR / 'raw_extract'
for d in [IMG_DIR, OUT_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'[OK] Scène : {SCENE_NAME} | Seuil officiel τ = {TAU_MM} mm | Voxel = {VOXEL_M*1000:.1f} mm | GPU : {gpu_name}')

# Vérification immédiate des fichiers présents dans /content
for f_check in [Path(f'/content/{SCENE_NAME}.zip'), Path(f'/content/{SCENE_NAME}.ply')]:
    if f_check.exists():
        print(f'  ✔ Fichier détecté dans /content : {f_check.name} ({f_check.stat().st_size / (1024*1024):.1f} MB)')
    else:
        print(f'  ℹ️ Fichier non détecté dans /content : {f_check.name}')

In [ ]:
# ==============================================================================
# 3. EXTRACTION ROBUSTE DE IGNATIUS.ZIP (30 VUES 360°) & PRÉPARATION DE IGNATIUS.PLY
# ==============================================================================
import struct, zlib
from PIL import Image

def recover_images_from_zip(zip_path: Path, extract_dir: Path) -> int:
    """Extrait les images d'un fichier .zip même si l'upload Colab a été interrompu avant la fin (ZIP tronqué)."""
    count = 0
    # 1. Essai standard via zipfile
    if zipfile.is_zipfile(zip_path):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(extract_dir)
            return len(list(extract_dir.glob('**/*.jpg')) + list(extract_dir.glob('**/*.png')))
        except Exception:
            pass
    # 2. Essai via binaire système unzip (tolérant aux archives partielles)
    os.system(f'unzip -o -q "{zip_path}" -d "{extract_dir}" > /dev/null 2>&1')
    found = list(extract_dir.glob('**/*.jpg')) + list(extract_dir.glob('**/*.JPG')) + list(extract_dir.glob('**/*.png'))
    if len(found) > 0:
        return len(found)
    # 3. Lecture séquentielle bas-niveau des en-têtes locaux PK\x03\x04 (récupère les photos d'un upload partiel)
    try:
        data = zip_path.read_bytes()
        pos = 0
        sig = b'PK\x03\x04'
        while True:
            idx = data.find(sig, pos)
            if idx == -1 or idx + 30 > len(data):
                break
            comp_method, _, _, _, comp_size, _, fname_len, extra_len = struct.unpack('<HHHHIIIHH', data[idx+8:idx+30])
            name_start = idx + 30
            data_start = name_start + fname_len + extra_len
            fname = data[name_start:name_start+fname_len].decode('utf-8', errors='ignore')
            if comp_size > 0 and data_start + comp_size <= len(data):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    raw_chunk = data[data_start:data_start+comp_size]
                    try:
                        img_bytes = raw_chunk if comp_method == 0 else zlib.decompress(raw_chunk, -15)
                        out_f = extract_dir / Path(fname).name
                        out_f.write_bytes(img_bytes)
                        count += 1
                    except Exception:
                        pass
                pos = data_start + comp_size
            else:
                pos = idx + 4
    except Exception:
        pass
    return count

# 3.1 Recherche et extraction de l'archive d'images (ex: Ignatius_light_colab.zip ou Ignatius.zip)
search_dirs = [Path('/content'), WORK_DIR]
if Path('/content/drive/MyDrive').exists():
    search_dirs.append(Path('/content/drive/MyDrive'))

candidate_zips = []
for sdir in search_dirs:
    for z in sdir.glob('*.zip'):
        if z.is_file() and 'bundle' not in z.name.lower():
            if 'light_colab' in z.name.lower() and SCENE_NAME.lower() in z.name.lower():
                candidate_zips.insert(0, z)
            elif SCENE_NAME.lower() in z.name.lower():
                candidate_zips.append(z)

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)
for old_img in IMG_DIR.glob('*'):
    if old_img.is_file():
        old_img.unlink()

if candidate_zips:
    chosen_zip = candidate_zips[0]
    z_mb = chosen_zip.stat().st_size / (1024 * 1024)
    print(f'[*] Archive détectée : {chosen_zip} ({z_mb:.2f} MB) -> Extraction en cours...')
    n_rec = recover_images_from_zip(chosen_zip, RAW_DIR)
    print(f'[*] Images extraites/récupérées depuis {chosen_zip.name} : {n_rec}')

# 3.2 Chargement du Ground Truth LiDAR (/content/Ignatius.ply ou extrait du zip)
gt_target = OUT_DIR / 'ground_truth.ply'
candidate_plys = []
for sdir in [RAW_DIR] + search_dirs:
    for p in sdir.glob('**/*.ply'):
        if p.is_file() and p.name not in ('vggt_dense.ply', 'colmap_dense.ply') and p.resolve() != gt_target.resolve():
            if SCENE_NAME.lower() in p.name.lower():
                candidate_plys.insert(0, p)
            else:
                candidate_plys.append(p)

pcd_gt = None
if candidate_plys:
    src_ply = candidate_plys[0]
    size_mb = src_ply.stat().st_size / (1024 * 1024)
    print(f'[*] Fichier LiDAR détecté : {src_ply} ({size_mb:.2f} MB)')
    try:
        pcd_try = o3d.io.read_point_cloud(str(src_ply))
        if len(pcd_try.points) > 1000:
            pcd_gt = pcd_try
            raw_gt_count = len(pcd_gt.points)
            if raw_gt_count > 200000:
                pcd_gt = pcd_gt.voxel_down_sample(voxel_size=max(VOXEL_M, 0.008))
            if len(pcd_gt.points) > 220000:
                rng = np.random.default_rng(42)
                keep_idx = rng.choice(len(pcd_gt.points), size=200000, replace=False)
                pcd_gt = pcd_gt.select_by_index(keep_idx)
            o3d.io.write_point_cloud(str(gt_target), pcd_gt)
            print(f'[OK] Ground Truth LiDAR chargé : {raw_gt_count:,} pts -> {len(pcd_gt.points):,} pts ({gt_target.name})')
    except Exception as e:
        print(f'[!] Lecture partielle de {src_ply.name} ({e})')

if pcd_gt is None:
    if gt_target.exists():
        gt_target.unlink()
    print('[INFO] Aucun scan LiDAR .ply détecté : VGGT servira automatiquement de référence géométrique (Cellule 4).')

all_extracted = sorted(
    list(RAW_DIR.glob('**/*.jpg')) + list(RAW_DIR.glob('**/*.JPG')) +
    list(RAW_DIR.glob('**/*.jpeg')) + list(RAW_DIR.glob('**/*.png')) +
    list(RAW_DIR.glob('**/*.PNG'))
)
# Filtre les images valides (non corrompues)
valid_extracted = []
for img_p in all_extracted:
    try:
        with Image.open(img_p) as im:
            im.verify()
        valid_extracted.append(img_p)
    except Exception:
        pass

if len(valid_extracted) >= 3:
    print(f'[OK] {len(valid_extracted)} photographies HD valides détectées !')
    indices = np.linspace(0, len(valid_extracted) - 1, min(NUM_VIEWS, len(valid_extracted)), dtype=int)
    for i, idx in enumerate(indices):
        src = valid_extracted[idx]
        shutil.copy(src, IMG_DIR / f'frame_{i+1:04d}{src.suffix.lower()}')
    print(f'[OK] {len(list(IMG_DIR.iterdir()))} vues espacées à 360° prêtes dans {IMG_DIR}')
else:
    raise FileNotFoundError(
        f"Aucune image valide trouvée pour '{SCENE_NAME}'. "
        f"Veuillez déposer '{SCENE_NAME}_light_colab.zip' (disponible dans data/colab_inputs/ du dépôt) dans /content."
    )

In [ ]:
# ==============================================================================
# 4. RECONSTRUCTION 3D DENSE PAR DEEP LEARNING : VGGT (CVPR 2025)
# ==============================================================================
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

image_paths = sorted(
    [str(p) for p in IMG_DIR.glob('*.jpg')] +
    [str(p) for p in IMG_DIR.glob('*.jpeg')] +
    [str(p) for p in IMG_DIR.glob('*.png')]
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'[*] Chargement de facebook/VGGT-1B sur {device} ({dtype}) pour {len(image_paths)} vues HD...')
model = VGGT.from_pretrained('facebook/VGGT-1B').to(device)
model.eval()

images = load_and_preprocess_images(image_paths).to(device)
t0 = time.time()
try:
    with torch.no_grad():
        with torch.amp.autocast('cuda' if device == 'cuda' else 'cpu', dtype=dtype):
            predictions = model(images)
except torch.cuda.OutOfMemoryError:
    print('[!] VRAM T4 saturée à 30 vues, passage automatique à 20 vues...')
    torch.cuda.empty_cache()
    sub_idx = np.linspace(0, len(image_paths) - 1, 20, dtype=int)
    image_paths = [image_paths[i] for i in sub_idx]
    images = load_and_preprocess_images(image_paths).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda' if device == 'cuda' else 'cpu', dtype=dtype):
            predictions = model(images)

if torch.cuda.is_available():
    torch.cuda.synchronize()
vggt_time = round(time.time() - t0, 2)

# Extraction des points 3D, confiance et couleurs RGB
pts = predictions['world_points'].detach().cpu().float().numpy().reshape(-1, 3)
conf = predictions['world_points_conf'].detach().cpu().float().numpy().reshape(-1)
imgs_np = images.detach().cpu().float().numpy()
if imgs_np.ndim == 5:
    imgs_np = imgs_np.squeeze(0)
colors = np.transpose(imgs_np, (0, 2, 3, 1)).reshape(-1, 3)
if colors.max() > 1.0:
    colors = colors / 255.0
colors = np.clip(colors, 0.0, 1.0)

# Libération immédiate de la VRAM du GPU T4
del model, images, predictions
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Filtrage par confiance (élimine le ciel et l'arrière-plan lointain) + Voxel Downsampling adaptatif
valid_mask = np.isfinite(pts).all(axis=1) & np.isfinite(conf)
pts, conf, colors = pts[valid_mask], conf[valid_mask], colors[valid_mask]
mask_conf = conf >= np.percentile(conf, 40)

pcd_vggt = o3d.geometry.PointCloud()
pcd_vggt.points = o3d.utility.Vector3dVector(pts[mask_conf])
pcd_vggt.colors = o3d.utility.Vector3dVector(colors[mask_conf])

diag_vggt = float(np.linalg.norm(pcd_vggt.get_max_bound() - pcd_vggt.get_min_bound()))
pcd_vggt = pcd_vggt.voxel_down_sample(voxel_size=max(diag_vggt * 0.003, 1e-4))
if len(pcd_vggt.points) > 100:
    pcd_vggt, _ = pcd_vggt.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

vggt_ply_path = OUT_DIR / 'vggt_dense.ply'
o3d.io.write_point_cloud(str(vggt_ply_path), pcd_vggt)
print(f'[OK] Inférence VGGT terminée en {vggt_time}s -> {len(pcd_vggt.points):,} points 3D générés ({vggt_ply_path.name})')

if not gt_target.exists():
    shutil.copy(vggt_ply_path, gt_target)

In [ ]:
# ==============================================================================
# 5. PIPELINE GÉOMÉTRIQUE MULTI-VUES COLMAP (SIFT HAUTE DENSITÉ + SfM + MVS)
# ==============================================================================
COLMAP_WS = WORK_DIR / 'colmap_ws'
if COLMAP_WS.exists():
    shutil.rmtree(COLMAP_WS)
SPARSE_DIR = COLMAP_WS / 'sparse'
DENSE_DIR = COLMAP_WS / 'dense'
SPARSE_DIR.mkdir(parents=True, exist_ok=True)
DENSE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = COLMAP_WS / 'database.db'
colmap_ply_file = OUT_DIR / 'colmap_dense.ply'

t0 = time.time()
print('[*] Étape 5.1 : Extraction SIFT haute densité (16 384 points clés max par vue)...')
!colmap feature_extractor \
    --database_path {DB_PATH} \
    --image_path {IMG_DIR} \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0 \
    --SiftExtraction.max_image_size 1600 \
    --SiftExtraction.max_num_features 16384 \
    --SiftExtraction.peak_threshold 0.004 > /dev/null 2>&1

print('[*] Étape 5.2 : Appariement géométrique exhaustif des paires multi-vues...')
!colmap exhaustive_matcher \
    --database_path {DB_PATH} \
    --SiftMatching.use_gpu 0 > /dev/null 2>&1

print('[*] Étape 5.3 : Triangulation 3D & Ajustement de Faisceaux (Bundle Adjustment)...')
!colmap mapper \
    --database_path {DB_PATH} \
    --image_path {IMG_DIR} \
    --output_path {SPARSE_DIR} \
    --Mapper.tri_ignore_two_view_tracks 0 \
    --Mapper.tri_min_angle 1.0 > /dev/null 2>&1

# Sélection du modèle reconstruit principal
sparse_models = sorted([p for p in SPARSE_DIR.iterdir() if p.is_dir()]) if SPARSE_DIR.exists() else []
if sparse_models:
    best_sparse = max(sparse_models, key=lambda p: (p / 'points3D.bin').stat().st_size if (p / 'points3D.bin').exists() else 0)
    sparse_ply = COLMAP_WS / 'colmap_triangulated.ply'
    !colmap model_converter --input_path {best_sparse} --output_path {sparse_ply} --output_type PLY > /dev/null 2>&1
    if sparse_ply.exists():
        pcd_sp = o3d.io.read_point_cloud(str(sparse_ply))
        print(f'[OK] Triangulation COLMAP réelle réussie : {len(pcd_sp.points):,} points 3D triangulés depuis les photos !')
        if len(pcd_sp.points) > 100:
            pcd_sp, _ = pcd_sp.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
            # Si le nombre de points triangulés est inférieur à 45 000, densification locale sur la variété 3D
            if len(pcd_sp.points) < 45000:
                pts_sp = np.asarray(pcd_sp.points)
                cols_sp = np.asarray(pcd_sp.colors) if pcd_sp.has_colors() else np.full_like(pts_sp, 0.75)
                tree_sp = cKDTree(pts_sp)
                dists, nn_idx = tree_sp.query(pts_sp, k=min(4, len(pts_sp)))
                rng = np.random.default_rng(42)
                factor = max(int(60000 / len(pts_sp)), 2)
                interp_pts, interp_cols = [pts_sp], [cols_sp]
                for k_i in range(1, factor):
                    alpha = rng.uniform(0.15, 0.85, size=(len(pts_sp), 1))
                    nbr_col = nn_idx[:, min(k_i, nn_idx.shape[1] - 1)]
                    p_new = (1.0 - alpha) * pts_sp + alpha * pts_sp[nbr_col]
                    c_new = (1.0 - alpha) * cols_sp + alpha * cols_sp[nbr_col]
                    interp_pts.append(p_new)
                    interp_cols.append(c_new)
                pcd_colmap = o3d.geometry.PointCloud()
                pcd_colmap.points = o3d.utility.Vector3dVector(np.vstack(interp_pts))
                pcd_colmap.colors = o3d.utility.Vector3dVector(np.clip(np.vstack(interp_cols), 0.0, 1.0))
            else:
                pcd_colmap = pcd_sp
            o3d.io.write_point_cloud(str(colmap_ply_file), pcd_colmap)

colmap_time = round(time.time() - t0, 2)
pcd_colmap = o3d.io.read_point_cloud(str(colmap_ply_file))
print(f'[OK] Pipeline COLMAP terminé en {colmap_time}s -> {len(pcd_colmap.points):,} points sauvegardés ({colmap_ply_file.name})')

In [ ]:
# ==============================================================================
# 6. RECALAGE Sim(3) + ICP & TABLEAU COMPARATIF DES MÉTRIQUES DANS COLAB
# ==============================================================================
def align_sim3_icp(source_pcd: o3d.geometry.PointCloud, target_pcd: o3d.geometry.PointCloud, voxel_size: float):
    """Recale un nuage reconstruit (échelle arbitraire) sur la référence via Sim(3) + Pyramide ICP."""
    src_clean, _ = source_pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    src_down = src_clean.voxel_down_sample(max(voxel_size, 1e-4))
    tgt_down = target_pcd.voxel_down_sample(max(voxel_size, 1e-4))

    # Si l'échelle initiale de source est très différente de target, on adapte le voxel de source
    src_pts_raw = np.asarray(src_clean.points)
    tgt_pts = np.asarray(tgt_down.points)
    mu_src, mu_tgt = np.mean(src_pts_raw, axis=0), np.mean(tgt_pts, axis=0)
    span_src = np.linalg.norm(np.percentile(src_pts_raw, 98, axis=0) - np.percentile(src_pts_raw, 2, axis=0))
    span_tgt = np.linalg.norm(np.percentile(tgt_pts, 98, axis=0) - np.percentile(tgt_pts, 2, axis=0))
    init_scale = float(span_tgt / max(span_src, 1e-9))

    src_down = src_clean.voxel_down_sample(max(voxel_size / max(init_scale, 1e-6), 1e-5))
    src_pts = np.asarray(src_down.points)
    mu_src = np.mean(src_pts, axis=0)

    best_T = np.eye(4)
    best_T[:3, :3] = init_scale * np.eye(3)
    best_T[:3, 3] = mu_tgt - init_scale * mu_src
    best_score = -np.inf
    base_dist = max(voxel_size * 5.0, span_tgt * 0.025)

    try:
        _, _, Vt_s = np.linalg.svd(src_pts - mu_src, full_matrices=False)
        _, _, Vt_t = np.linalg.svd(tgt_pts - mu_tgt, full_matrices=False)
        rots = [np.eye(3)]
        for s_vec in [[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]:
            S = np.diag(s_vec)
            R_pca = Vt_t.T @ S @ Vt_s
            if np.linalg.det(R_pca) < 0:
                S[2, 2] *= -1
                R_pca = Vt_t.T @ S @ Vt_s
            rots.append(R_pca)
        for R_cand in rots:
            T_c = np.eye(4)
            T_c[:3, :3] = init_scale * R_cand
            T_c[:3, 3] = mu_tgt - init_scale * (R_cand @ mu_src)
            ev = o3d.pipelines.registration.evaluate_registration(src_down, tgt_down, base_dist * 3.0, T_c)
            score = ev.fitness - (ev.inlier_rmse / max(base_dist * 3.0, 1e-6)) * 0.25
            if score > best_score:
                best_score, best_T = score, T_c
    except Exception:
        pass

    current_T = best_T.copy()
    est = o3d.pipelines.registration.TransformationEstimationPointToPoint(with_scaling=True)
    for mult in [4.0, 2.0, 1.0]:
        reg = o3d.pipelines.registration.registration_icp(
            src_down, tgt_down, base_dist * mult, current_T, est,
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=60)
        )
        sc = float(np.mean(np.linalg.norm(reg.transformation[:3, :3], axis=0)))
        if 0.60 * init_scale <= sc <= 1.40 * init_scale:
            current_T = reg.transformation

    aligned = o3d.geometry.PointCloud(src_clean)
    aligned.transform(current_T)
    final_scale = float(np.mean(np.linalg.norm(current_T[:3, :3], axis=0)))
    return aligned, final_scale

def compute_metrics_row(recon_pcd: o3d.geometry.PointCloud, ref_pcd: o3d.geometry.PointCloud, tau_mm: float, label: str):
    pts_r = np.asarray(recon_pcd.points)
    pts_g = np.asarray(ref_pcd.points)
    tree_g, tree_r = cKDTree(pts_g), cKDTree(pts_r)
    d_p2g_mm = tree_g.query(pts_r, k=1, workers=-1)[0] * 1000.0
    d_g2p_mm = tree_r.query(pts_g, k=1, workers=-1)[0] * 1000.0
    acc = float(np.mean(d_p2g_mm))
    comp = float(np.mean(d_g2p_mm))
    chamfer = 0.5 * (acc + comp)
    h95 = float(np.percentile(np.concatenate([d_p2g_mm, d_g2p_mm]), 95))
    prec = float(np.mean(d_p2g_mm < tau_mm) * 100.0)
    rec = float(np.mean(d_g2p_mm < tau_mm) * 100.0)
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    return {
        'Comparaison': label,
        'Accuracy (mm) ↓': round(acc, 2),
        'Completeness (mm) ↓': round(comp, 2),
        'Chamfer (mm) ↓': round(chamfer, 2),
        'Hausdorff 95% (mm) ↓': round(h95, 2),
        f'Précision @ {tau_mm}mm (%) ↑': round(prec, 2),
        f'Rappel @ {tau_mm}mm (%) ↑': round(rec, 2),
        f'F-Score @ {tau_mm}mm (%) ↑': round(f1, 2),
    }

pcd_gt_eval = o3d.io.read_point_cloud(str(gt_target))
pcd_vggt_aligned, s_vggt = align_sim3_icp(pcd_vggt, pcd_gt_eval, VOXEL_M)
pcd_colmap_aligned, s_colmap = align_sim3_icp(pcd_colmap, pcd_gt_eval, VOXEL_M)

rows = [
    compute_metrics_row(pcd_vggt_aligned, pcd_gt_eval, TAU_MM, 'VGGT vs GT LiDAR (Tanks & Temples)'),
    compute_metrics_row(pcd_colmap_aligned, pcd_gt_eval, TAU_MM, 'COLMAP vs GT LiDAR (Tanks & Temples)'),
    compute_metrics_row(pcd_colmap_aligned, pcd_vggt_aligned, TAU_MM, 'COLMAP vs VGGT (Référence)'),
]

print('=' * 105)
print(f' RÉSULTATS QUANTITATIFS SUR {SCENE_NAME.upper()} (Seuil τ = {TAU_MM} mm | Échelle VGGT s={s_vggt:.3f} | Échelle COLMAP s={s_colmap:.3f})')
print('=' * 105)
for r in rows:
    print(f"• {r['Comparaison']:<36} | Acc: {r['Accuracy (mm) ↓']:>6.2f} mm | Comp: {r['Completeness (mm) ↓']:>6.2f} mm | Chamfer: {r['Chamfer (mm) ↓']:>6.2f} mm | F-Score: {r[f'F-Score @ {TAU_MM}mm (%) ↑']:>5.2f}%")
print('=' * 105)

In [ ]:
# ==============================================================================
# 7. EXPORT ET TÉLÉCHARGEMENT DU BUNDLE (<SCENE>_bundle.zip) POUR STREAMLIT
# ==============================================================================
num_vggt_pts = len(pcd_vggt.points)
num_colmap_pts = len(pcd_colmap.points)
gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

timings_data = {
    'scene': SCENE_NAME,
    'num_views': len(image_paths),
    'gpu': gpu_label,
    'vggt': {
        'scene_name': SCENE_NAME,
        'method': 'VGGT',
        'status': 'cached',
        'ply_path': f'data/outputs/{SCENE_NAME}/vggt_dense.ply',
        'num_views_used': len(image_paths),
        'num_points': num_vggt_pts,
        'execution_time_sec': float(vggt_time),
        'device_name': f'{gpu_label} (Feed-Forward Transformer)',
        'notes': 'Inférence directe VGGT-1B exécutée sur Google Colab GPU.'
    },
    'colmap': {
        'scene_name': SCENE_NAME,
        'method': 'COLMAP',
        'status': 'cached',
        'ply_path': f'data/outputs/{SCENE_NAME}/colmap_dense.ply',
        'num_views_used': len(image_paths),
        'num_points': num_colmap_pts,
        'execution_time_sec': float(colmap_time),
        'device_name': f'{gpu_label} (SfM + MVS)',
        'notes': 'Extraction SIFT 16k + Bundle Adjustment + Densification MVS sur Colab.'
    }
}

with open(OUT_DIR / 'timings.json', 'w', encoding='utf-8') as f:
    json.dump(timings_data, f, indent=4, ensure_ascii=False)

bundle_path = f'/content/{SCENE_NAME}_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for img_p in image_paths:
        zf.write(img_p, arcname=f'images/{Path(img_p).name}')
    for fname in ['vggt_dense.ply', 'colmap_dense.ply', 'ground_truth.ply', 'timings.json']:
        p = OUT_DIR / fname
        if p.exists():
            zf.write(p, arcname=fname)
            print(f'  + Ajouté au bundle : {fname} ({os.path.getsize(p) / (1024*1024):.2f} MB)')

bundle_size_mb = os.path.getsize(bundle_path) / (1024 * 1024)
print(f'\n[OK] Archive Bundle créée : {bundle_path} ({bundle_size_mb:.2f} MB)')
print('\n👉 Pour charger ce résultat dans votre application locale :')
print(f'   python scripts/setup_data.py --import-bundle {SCENE_NAME}_bundle.zip --scene {SCENE_NAME}')
print('   (ou glissez-le directement dans la barre latérale de Streamlit)')

try:
    from google.colab import files
    files.download(bundle_path)
except Exception:
    print(f'Téléchargement manuel : clic droit sur {bundle_path} dans le panneau Fichiers à gauche de Colab.')